In [1]:
#Step 1: Import all necessary libraries. In this case, we will need pytorch as well as the one-hot encoding.
import torch
import torch.nn.functional as F

In [2]:
#Step 2: Open the names dataset, store it as a variable inside our program
names = open('names.txt', 'r').read().splitlines()

#Quick Sanity Check
print(len(names)) #Should output 32033

32033


In [3]:
#We will need 2 arrays right now. xs and ys. They will store all the characters in names. 
#xs will store the first character, and ys will store the second character. 

xs, ys = [], [] 

#We will also need an StringToInteger Hashmap as well as an IntegerToString Hashmap

chars = sorted(set(''.join(names))) #This takes in all the characters that exist in names and sorts them alphabetically

#Sanity Check
print(len(chars)) # Should output : 26

stoi = {ch: i + 1 for i, ch in enumerate(chars)} #stoi is short for StringToInteger. 
stoi['.'] = 0 #We are going to manually add our '.' character inside our stoi
itos = {i : ch for ch, i in stoi.items()} #itos is short for IntegerToString



26


In [4]:
#Step 3: We will now loop through all of our names and assign them to our xs,ys arrays.

for n in names:
    chs = ['.'] + list(n) + ['.'] #We turn it into a list eg. ['.','e','m','m','a','.'] 
    for ch1, ch2 in zip(chs, chs[1:]): #Each iteration gives one bigram: ch1 is the current char, ch2 is the next char
        ix1 = stoi[ch1] 
        ix2 = stoi[ch2]
        xs.append(ix1)
        ys.append(ix2)

#Convert to tensor datatype
xs = torch.tensor(xs)
ys = torch.tensor(ys)

#Check the amount of examples in xs
num_examples = xs.nelement()

print("Examples: ", num_examples)

Examples:  228146


In [5]:
# 27x27 matrix of randomly initialized weights.
# requires_grad=True tells PyTorch to track operations on W so it can 
# automatically compute gradients (dLoss/dW) when we call loss.backward()
W = torch.randn((27,27) , requires_grad=True) #require_grad let's us modify the gradient. 

In [13]:
#Step 4: We can begin our forward descent and begin improving our loss value

loss = float('inf')

while loss > 2.5:

    # [FORWARD PASS]

    #Encode our xs with one-hot encoding, turns single value into a 1x27 vector where
    #   the value in xs is = 1 and the rest are 0
    #We must use float because ...
    # 1. Matrix multiplication (xenc @ W) requires both tensors to share the same dtype,
    #    and W is already a float tensor.
    # 2. Gradients (needed for backprop) can only flow through floating-point tensors,
    #    not integer tensors.
    xenc = F.one_hot(xs, num_classes=27).float()
    logits = xenc @ W #The raw output score. Must be processed further
    counts = logits.exp() #We use the .exp() to make all values positive as well as keep their relative positions
    prob = counts / counts.sum(1,keepdim=True) #Turns the counts into a probability the sums to 1

    #We then can calculate the loss be taking the probability 
    loss = -prob[torch.arange(num_examples), ys].log().mean() 

    # [BACKWARD PASS]
    W.grad = None
    loss.backward()

    #Train/Update 
    W.data -= 1 * W.grad

print(loss.item())


2.4548449516296387
